# Symptom-to-Diagnosis — Free-Text Classifier Analysis
## The live ML pillar of the Hybrid Medical Assistant

This notebook documents the **free-text symptom classifier** that runs on the live
path of the assistant. Given a natural-language symptom description (e.g. *"I've had
a burning feeling when I pee and I need to go constantly"*), it ranks likely
conditions and feeds that signal into the RAG + LLM pipeline.

### What this is (and how it differs from the XGBoost model)

The companion notebook [`ml_model_analysis.ipynb`](ml_model_analysis.ipynb) covers a
DDXPlus **XGBoost** classifier trained on *structured* evidence vectors. That model
is excellent on its own test set but is **off the live path**: in a chat app it only
ever sees a few symptoms parsed from free text (everything else marked *absent*),
which is a distribution it never saw in training → out-of-distribution → confidently
wrong (e.g. flu-like symptoms → "Tuberculosis 91%").

This model fixes that mismatch: it **trains and serves on the same thing — free
text**. So its held-out metrics actually predict live behaviour.

### What this notebook covers
1. **EDA** on `gretelai/symptom_to_diagnosis` (853 train / 212 test, 22 conditions).
2. The **embedding → logistic-regression** pipeline (the exact one that ships).
3. **Evaluation** — top-1 / top-3 accuracy, macro-F1, a per-class report, and a
   confusion matrix.
4. An **inference demo** on free-text inputs.
5. Honest **limitations**.

### Relationship to the main project
This reproduces exactly what ships in `ml_model/text_train.py` (training) and
`ml_model/text_predict.py` (serving). Two design choices make it a genuinely
*hybrid* component:

- It embeds text with **the same `S-PubMedBert` encoder the RAG retriever uses**, so
  the ML and retrieval layers share one representation.
- At serve time its top predictions **widen the RAG recall query** and are
  **cross-checked against the retrieved passages** (unsupported guesses are hidden),
  and it **abstains** when its top probability is below a confidence floor.

It is a *supplementary* signal — the grounded, cited LLM answer is authoritative.

## 1. Environment Setup

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data + model
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 90)

# The exact encoder the RAG retriever uses (biomedical sentence embeddings).
EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
DATASET = "gretelai/symptom_to_diagnosis"

## 2. Loading the dataset

### Data schema
`gretelai/symptom_to_diagnosis` is a small, freely available (no Kaggle auth)
dataset of first-person symptom descriptions paired with a diagnosis:

| column | meaning |
|--------|---------|
| `input_text` | a natural-language symptom description |
| `output_text` | the diagnosis label (one of 22 conditions) |

In [ ]:
ds = load_dataset(DATASET)
train_df = ds["train"].to_pandas().rename(columns={"input_text": "text", "output_text": "label"})
test_df = ds["test"].to_pandas().rename(columns={"input_text": "text", "output_text": "label"})

print(f"train: {len(train_df)} rows | test: {len(test_df)} rows | classes: {train_df.label.nunique()}")
train_df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Class distribution (train)
counts = train_df.label.value_counts()
print("classes:", len(counts))
print("min/median/max per class:", counts.min(), int(counts.median()), counts.max())
counts.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
counts.sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_title("Training examples per condition")
ax.set_xlabel("count")
plt.tight_layout()
plt.show()

### Observations — class distribution
The 22 conditions are **mildly imbalanced** (a few dozen examples each). It is small
data by deep-learning standards, which is exactly why we *don't* fine-tune a network:
we reuse a strong pre-trained biomedical encoder and train only a lightweight linear
head on top. `class_weight="balanced"` compensates for the imbalance.

In [ ]:
# Text length distribution (in words)
train_df["n_words"] = train_df.text.str.split().str.len()
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(train_df.n_words, bins=30, ax=ax, color="#55A868")
ax.set_title("Symptom-description length (words)")
ax.set_xlabel("words per description")
plt.tight_layout()
plt.show()
train_df.n_words.describe().round(1)

In [ ]:
# A few example descriptions per condition
for lab in list(counts.index[:3]):
    print(f"\n=== {lab} ===")
    for t in train_df[train_df.label == lab].text.head(2):
        print(" -", t[:120])

## 4. The Embedding Pipeline

### Why embeddings (train == serve, shared encoder)
Each description is turned into a single 768-d vector by `S-PubMedBert`. This is the
crux of the design:

- **Train == serve.** Both training rows and live user messages are the same kind of
  object — free text — encoded the same way. No fragile symptom parsing, no fabricated
  "absent" features, so the model is *in-distribution* at serve time.
- **Shared representation.** The retriever embeds passages with this same encoder, so
  the ML signal lives in the same space as retrieval — a genuinely coupled hybrid.

Embeddings are L2-normalized (matches the shipped `rag.embeddings.EmbeddingModel`).

In [ ]:
encoder = SentenceTransformer(EMBEDDING_MODEL)

def embed(texts):
    return encoder.encode(list(texts), batch_size=32, convert_to_numpy=True,
                          normalize_embeddings=True, show_progress_bar=True).astype("float32")

X_train = embed(train_df.text)
X_test = embed(test_df.text)
print("embedding shape:", X_train.shape)  # (n, 768)

In [ ]:
# Encode labels to integer ids (sorted, matching ml_model/text_train.py)
labels = sorted(train_df.label.unique())
label_to_idx = {lab: i for i, lab in enumerate(labels)}
y_train = train_df.label.map(label_to_idx).to_numpy()
y_test = test_df.label.map(label_to_idx).to_numpy()
print(len(labels), "classes")

## 5. Model Training

### Why logistic regression on embeddings
A strong pre-trained encoder does the heavy lifting, so a **multinomial logistic
regression** head is enough to separate the 22 classes — and it's fast, tiny,
CPU-friendly, and gives calibrated-enough probabilities for the top-k differential
and the serve-time abstention threshold. This is the exact configuration in
`ml_model/text_train.py`.

In [ ]:
clf = LogisticRegression(max_iter=3000, C=10.0, class_weight="balanced")
clf.fit(X_train, y_train)
print("trained on", X_train.shape[0], "examples,", len(labels), "classes")

## 6. Evaluation

### Metrics and why each matters
- **Top-1 accuracy** — is the single best guess correct?
- **Top-3 accuracy** — is the true condition among the top 3? (The app shows a
  ranked short-list, so this reflects real usefulness.)
- **Macro-F1** — treats all 22 classes equally, so rare conditions aren't ignored.

Crucially, these are on a **held-out test split of the same free-text distribution**
the model serves — so unlike the DDXPlus model's test score, this number transfers to
production.

In [ ]:
proba = clf.predict_proba(X_test)
y_pred = proba.argmax(axis=1)

def topk_accuracy(proba, y_true, k):
    topk = np.argsort(proba, axis=1)[:, -k:]
    return np.mean([yt in row for yt, row in zip(y_true, topk)])

print(f"top-1 accuracy: {topk_accuracy(proba, y_test, 1):.3f}")
print(f"top-3 accuracy: {topk_accuracy(proba, y_test, 3):.3f}")
print(f"macro F1:       {f1_score(y_test, y_pred, average='macro'):.3f}")

### Per-class metrics

In [ ]:
report = classification_report(y_test, y_pred, target_names=labels, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.loc[labels].sort_values("f1-score", ascending=False).round(3)

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Confusion matrix (held-out test)")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Reading the confusion matrix
A strong diagonal means most conditions are classified correctly. Off-diagonal
clusters are usually **clinically plausible confusions** — conditions that genuinely
share symptom language (e.g. overlapping infectious or gastrointestinal presentations).
That is the honest failure mode to expect from symptom text alone, and it's exactly
why the model is a *pre-ranking signal* rather than a diagnosis: the LLM re-reasons
over retrieved literature and the app cross-checks each prediction against it.

## 7. Inference Demo (end-to-end)

In [ ]:
def predict_text(text, k=3):
    v = encoder.encode([text], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    p = clf.predict_proba(v)[0]
    top = np.argsort(p)[-k:][::-1]
    return [(labels[i], float(p[i])) for i in top]

examples = [
    "I have a burning feeling when I urinate and I need to go very often",
    "severe headache with nausea and sensitivity to light",
    "itchy red rash that is spreading on my arm",
    "my neck and back hurt and my arms feel numb and tingly",
]
for ex in examples:
    print(ex)
    for disease, prob in predict_text(ex):
        print(f"    {prob*100:5.1f}%  {disease}")
    print()

## 8. Saving Artifacts
This mirrors `ml_model/text_train.py`, which writes three files to
`ml_model/artifacts/` (gitignored): the classifier, the label list, and a metadata
sidecar recording the embedder + metrics so serving can refuse on an embedder
mismatch. Uncomment to write them from this notebook.

In [ ]:
# import json, joblib
# from pathlib import Path
# art = Path("../ml_model/artifacts"); art.mkdir(exist_ok=True)
# joblib.dump(clf, art / "symptom_text_clf.joblib")
# (art / "symptom_text_labels.json").write_text(json.dumps(labels, indent=2))
# (art / "symptom_text_meta.json").write_text(json.dumps({
#     "dataset": DATASET, "embedding_model": EMBEDDING_MODEL,
#     "top1_accuracy": round(float(topk_accuracy(proba, y_test, 1)), 4),
#     "top3_accuracy": round(float(topk_accuracy(proba, y_test, 3)), 4),
#     "macro_f1": round(float(f1_score(y_test, y_pred, average="macro")), 4),
#     "n_train": int(len(train_df)), "n_test": int(len(test_df)), "n_classes": len(labels),
# }, indent=2))
print("(saving is commented out — run `python -m ml_model.text_train` to produce the shipped artifacts)")

## 9. Summary and Limitations

### What this model does
Reads a free-text symptom description, embeds it with a biomedical sentence encoder,
and ranks the 22 conditions it was trained on. Because train == serve, its held-out
top-1 (~0.9) reflects live behaviour, and it is in-distribution on real user messages.

### Known limitations (honest — this matters for a medical project)
- **Closed set of 22 conditions.** Anything outside the training label set can't be
  predicted correctly; the app's **abstention threshold** (hide the ML when the top
  probability is low) is what stops it from confidently guessing on out-of-scope text.
- **Small dataset (~1k rows).** Metrics have real variance; treat them as directional.
  The linear head keeps overfitting in check, but more data would help the rarer classes.
- **Symptom text is inherently ambiguous** — clinically overlapping conditions are
  confused (see the confusion matrix). This is a *pre-ranking* aid, **not a diagnosis**.
- **Encoder dependency.** The classifier lives in `S-PubMedBert`'s embedding space; it
  must be served with the same encoder (the shipped model guards against a mismatch).

### How this connects to the production system
`ml_model/text_predict.py` loads these artifacts and serves `predict_text`. In
`assistant.prepare` the top predictions widen the RAG recall query, are cross-checked
against the retrieved passages, and are shown only when supported — while the grounded,
cited LLM answer remains the authoritative output. Not a medical device; educational use only.